# Notebook 04 — Customer Segmentation

**Project:** CX Intelligence — NPS & Sentiment Analysis  
**Author:** Nicolás Zuleta Sierra

## Objectives
1. Build a feature matrix from NLP-derived features
2. Select optimal k with Elbow method + Silhouette score
3. Train K-Means and assign customers to segments
4. Profile each segment statistically
5. Name segments with CX-actionable labels
6. Visualize clusters in 2D (PCA) with Plotly
7. Provide CX action recommendations per segment

**Input:** `data/processed/features_nlp.csv` (with topic columns from Notebook 03)  
**Output:** `models/kmeans_model.joblib` + `cluster` column in features_nlp.csv

## 0. Imports & Configuration

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder, StandardScaler

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from src.segmentation import (
    assign_clusters,
    build_cluster_profiles,
    build_feature_matrix,
    find_optimal_k,
    name_clusters,
    save_kmeans_model,
    train_kmeans,
)

plt.rcParams["figure.figsize"] = (12, 5)
sns.set_style("whitegrid")
pd.set_option("display.max_colwidth", 80)

FEATURES_PATH = ROOT / "data" / "processed" / "features_nlp.csv"
KMEANS_MODEL_PATH = ROOT / "models" / "kmeans_model.joblib"

print(f"Features exist: {FEATURES_PATH.exists()}")

## 1. Load Features Dataset

In [ ]:
df = pd.read_csv(FEATURES_PATH, low_memory=False)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# Verify required columns
required = ["finbert_score", "dominant_topic", "nps_score", "product"]
missing = [c for c in required if c not in df.columns]
if missing:
    print(f"WARNING — Missing columns: {missing}")
    print("Run Notebooks 01, 02, and 03 first.")
else:
    print("All required columns present.")
df.head(2)

## 2. Build Feature Matrix

Features selected for clustering:
- **finbert_score** — FinBERT confidence score (0–1)
- **dominant_topic** — LDA topic index (ordinal proxy)
- **nps_score** — Simulated satisfaction score (0–10)
- **product_encoded** — Banking product (label-encoded)

All features are standardized (z-score) before clustering.

In [ ]:
# Drop rows with any missing feature
df_cluster = df.dropna(subset=["finbert_score", "dominant_topic", "nps_score"]).copy()
print(f"Rows available for clustering: {len(df_cluster):,}")

X, le = build_feature_matrix(df_cluster)
print(f"Feature matrix shape: {X.shape}")

## 3. Elbow Method

In [ ]:
print("Computing Elbow + Silhouette for k=2 to 8…")
k_results = find_optimal_k(X, k_range=range(2, 9))

elbow_df = pd.DataFrame(
    [{"k": k, "inertia": v} for k, v in k_results["elbow"].items()]
)

fig = px.line(
    elbow_df, x="k", y="inertia",
    markers=True,
    title="Elbow Method — Within-Cluster Sum of Squares (WCSS)",
    labels={"k": "Number of Clusters (k)", "inertia": "WCSS (Inertia)"},
    color_discrete_sequence=["#1e40af"],
)
fig.show()

## 4. Silhouette Score

In [ ]:
silhouette_df = pd.DataFrame(
    [{"k": k, "silhouette": v} for k, v in k_results["silhouette"].items()]
)

fig = px.bar(
    silhouette_df, x="k", y="silhouette",
    color="silhouette",
    color_continuous_scale=["#fee2e2", "#22c55e"],
    title="Silhouette Score by k",
    labels={"k": "Number of Clusters (k)", "silhouette": "Silhouette Score"},
    text="silhouette",
)
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(coloraxis_showscale=False)
fig.show()

optimal_k = silhouette_df.loc[silhouette_df["silhouette"].idxmax(), "k"]
print(f"Optimal k (max silhouette): {optimal_k}")

## 5. Train K-Means

In [ ]:
N_CLUSTERS = int(optimal_k)  # Override here if needed
print(f"Training K-Means with k={N_CLUSTERS}…")

kmeans = train_kmeans(X, n_clusters=N_CLUSTERS)
df_cluster = assign_clusters(df_cluster, kmeans, X)

print("Cluster distribution:")
print(df_cluster["cluster"].value_counts().sort_index())

## 6. Cluster Profiles

In [ ]:
profiles = build_cluster_profiles(df_cluster)
print("=== CLUSTER PROFILES ===")
print(profiles.to_string(index=False))

## 7. CX Segment Naming

Clusters are ranked by average NPS and assigned a CX-actionable name.  
Names reflect both the data profile and the recommended intervention.

In [ ]:
cx_name_map = name_clusters(profiles)

print("\n=== CX SEGMENT NAMES ===")
for cluster_id, name in sorted(cx_name_map.items()):
    row = profiles[profiles["cluster"] == cluster_id].iloc[0]
    print(f"Cluster {cluster_id} → {name}")
    print(f"  NPS: {row['avg_nps']} | Sentiment: {row['dominant_sentiment']} | "
          f"Detractors: {row['pct_detractors']}% | n={row['n_complaints']:,}")

df_cluster["cluster_name"] = df_cluster["cluster"].map(cx_name_map)

## 8. PCA Scatter Plot — 2D Cluster Visualization

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

df_cluster["pca_1"] = X_pca[:, 0]
df_cluster["pca_2"] = X_pca[:, 1]

variance_explained = pca.explained_variance_ratio_
print(f"Variance explained by PCA: {variance_explained[0]:.1%} + {variance_explained[1]:.1%} = {sum(variance_explained):.1%}")

fig = px.scatter(
    df_cluster.sample(min(5000, len(df_cluster)), random_state=42),  # sample for speed
    x="pca_1", y="pca_2",
    color="cluster_name",
    opacity=0.5,
    title=f"Customer Segments — PCA Projection (k={N_CLUSTERS})",
    labels={
        "pca_1": f"PC1 ({variance_explained[0]:.1%} variance)",
        "pca_2": f"PC2 ({variance_explained[1]:.1%} variance)",
    },
    hover_data=["nps_score", "finbert_label", "product"] if all(
        c in df_cluster.columns for c in ["nps_score", "finbert_label", "product"]
    ) else None,
)
fig.update_layout(height=550, legend_title="CX Segment")
fig.show()

## 9. Product Distribution by Segment

In [ ]:
if "product" in df_cluster.columns:
    prod_dist = (
        df_cluster.groupby(["cluster_name", "product"])
        .size()
        .reset_index(name="count")
    )
    fig = px.bar(
        prod_dist, x="cluster_name", y="count", color="product",
        barmode="stack",
        title="Product Distribution by CX Segment",
        labels={"count": "Complaints", "cluster_name": "Segment"},
    )
    fig.update_layout(xaxis_tickangle=-20, height=450)
    fig.show()

## 10. CX Action Recommendations

In [ ]:
cx_recommendations = {
    "Critical Risk": (
        "Immediate personal outreach within 48h. Escalate to specialized retention team. "
        "Assign a dedicated case manager. High churn probability."
    ),
    "Silent Dissatisfied": (
        "Proactive NPS survey recommended. Neutral language masks real frustration. "
        "Targeted recovery campaign. Do not ignore — this segment churns quietly."
    ),
    "Neutral Observers": (
        "Monitor for sentiment drift. Opportunity for proactive engagement. "
        "Educational content and proactive service updates can move this group toward Promoters."
    ),
    "Promoter Candidates": (
        "Activate for referral programs. Request public reviews on Google/Trustpilot. "
        "Cross-sell premium products. Low intervention cost, high ROI."
    ),
    "Active Promoters": (
        "Brand ambassador program. Referral incentives. Minimal service intervention needed. "
        "Maintain satisfaction — avoid disrupting what is working."
    ),
}

print("=== CX ACTION RECOMMENDATIONS ===")
for segment, rec in cx_recommendations.items():
    if segment in cx_name_map.values():
        print(f"\n[{segment}]")
        print(f"  {rec}")

## 11. Save Model and Updated Dataset

In [ ]:
# Save K-Means model
save_kmeans_model(kmeans, KMEANS_MODEL_PATH)

# Merge cluster results back into features_nlp.csv
cluster_cols = ["complaint_id", "cluster", "cluster_name", "pca_1", "pca_2"]
df_full = pd.read_csv(FEATURES_PATH, low_memory=False)
df_cluster_export = df_cluster[cluster_cols].copy()

# Drop existing cluster columns if re-running
for col in ["cluster", "cluster_name", "pca_1", "pca_2"]:
    if col in df_full.columns:
        df_full = df_full.drop(columns=[col])

df_full = df_full.merge(df_cluster_export, on="complaint_id", how="left")
df_full.to_csv(FEATURES_PATH, index=False)
print(f"Updated features_nlp.csv with cluster assignments: {len(df_full):,} rows")
df_full[["complaint_id", "nps_score", "finbert_label", "cluster", "cluster_name"]].head()

## 12. Summary

*(Fill in with actual segment profiles after running)*

| Segment | Avg NPS | Dominant Sentiment | Top Product | % Detractors | n |
|---------|---------|-------------------|-------------|--------------|---|
| Critical Risk | XX | negative | XX | XX% | XX |
| Silent Dissatisfied | XX | neutral | XX | XX% | XX |
| Neutral Observers | XX | neutral | XX | XX% | XX |
| Promoter Candidates | XX | positive | XX | XX% | XX |
| Active Promoters | XX | positive | XX | XX% | XX |

**Key insight:** The 'Silent Dissatisfied' segment is the most dangerous —
neutral language combined with low NPS means these customers are at churn risk
but will not escalate further. Standard sentiment models miss them.
This is only detectable by combining NPS simulation + FinBERT + segmentation.